# Casting Quality Inspection
## Architecture and Design Decisions

### Binary Image Classification Using Deep Learning

**Objective:**
Develop a deep learning system to classify casting product images into two categories:

- **Class 0 → Non-defective**
- **Class 1 → Defective**

### Project Approach

This project follows a progressive deep learning approach:

1. Dataset preparation and preprocessing
2. Custom CNN baseline
3. Model training and evaluation
4. Transfer learning using MobileNetV2
5. Fine-tuning
6. Threshold analysis
7. Grad-CAM explainability
8. Streamlit deployment

The custom CNN serves as the baseline architecture required for the
assessment. MobileNetV2, fine-tuning, threshold optimization,
Grad-CAM, and Streamlit deployment are additional improvements
implemented in the project.

# 1. Dataset

The project uses the **Casting Product Image Data for Quality Inspection**
dataset for binary image classification.

The dataset contains two categories of casting images:

| Label | Class | Description |
|---|---|---|
| 0 | Non-defective | Casting product without a visible defect |
| 1 | Defective | Casting product containing a defect |

### Dataset Organization

The images are organized into folders representing the two classes:

- `ok_front` → Non-defective
- `def_front` → Defective

The objective is to learn visual patterns that allow the model to
distinguish between defective and non-defective casting products.

# 2. Input Representation

## Image Size

The input images are resized to:

**224 × 224 × 3**

where:

- `224` = image height
- `224` = image width
- `3` = RGB color channels

### Why 224 × 224?

A fixed input size is required so that all images can be processed
by the neural network using a consistent tensor shape.

The size of **224 × 224** provides a practical balance between:

- Image detail
- Computational cost
- Memory requirements
- Compatibility with the MobileNetV2 architecture used later

### Pixel Representation

The original image pixels are represented in the range:

**0–255**

During preprocessing, the values are normalized to:

**0–1**

This provides a suitable numerical range for neural network training.

# 3. Data Augmentation

Data augmentation is applied during training to expose the model
to slightly different versions of the same images.

The augmentation operations used in the project include:

- Horizontal flipping
- Small rotations
- Zoom
- Contrast variation

### Purpose of Data Augmentation

Data augmentation helps the model become less dependent on the
exact appearance or orientation of individual training images.

It can improve:

- Generalization
- Robustness to image variation
- Resistance to overfitting

The augmented images are used during training, while validation
and test images are kept separate for unbiased evaluation.

# 4. Baseline CNN Architecture

The baseline model follows the CNN architecture specified for
the assessment.

The purpose of the baseline is to understand how a conventional
CNN performs on the casting defect classification task before
introducing transfer learning.

## Complete Baseline Architecture

```text
Input Image
224 × 224 × 3
       ↓
Data Augmentation
       ↓
Rescaling
0–255 → 0–1
       ↓
Conv2D
32 filters, 3×3 kernel, ReLU
       ↓
MaxPooling2D
       ↓
Conv2D
64 filters, 3×3 kernel, ReLU
       ↓
MaxPooling2D
       ↓
Conv2D
128 filters, 3×3 kernel, ReLU
       ↓
MaxPooling2D
       ↓
GlobalAveragePooling2D
       ↓
Dropout
0.40
       ↓
Dense
64 neurons, ReLU
       ↓
Dense
1 neuron, Sigmoid
       ↓
Defective Probability

Raw Image
   ↓
Edges and simple patterns
   ↓
Intermediate visual patterns
   ↓
Complex visual patterns
   ↓
Compact feature representation
   ↓
Binary classification


## 4.1 Convolutional Layer — 32 Filters

The first convolutional layer uses:

**32 filters with a 3 × 3 kernel**

Conceptually:

```python
Conv2D(32, 3, activation="relu")


## 4.2 Convolutional Layer — 64 Filters

The second convolutional layer uses:

**64 filters with a 3 × 3 kernel**

Conceptually:

```python
Conv2D(64, 3, activation="relu")





## 4.3 Convolutional Layer — 128 Filters

The third convolutional layer uses:

**128 filters with a 3 × 3 kernel**
Conceptually:

```python
Conv2D(128, 3, activation="relu")



## 4.4 ReLU Activation

The convolutional layers and hidden Dense layer use the
**ReLU (Rectified Linear Unit)** activation function.

Conceptually:

```python
activation="relu"

## 4.7 Dropout

The baseline architecture uses:

```python
Dropout(0.40)

## 4.5 MaxPooling

MaxPooling is applied after each convolutional block.

Conceptually:

```python
MaxPooling2D()

## 4.8 Dense Layer

After the convolutional feature extraction stage, the model
uses a fully connected Dense layer:

```python
Dense(64, activation="relu")



The assessment specifies a single sigmoid output and an initial 0.50 threshold. :contentReference[oaicite:9]{index=9}

---

# CELL 15 — COMPLETE ARCHITECTURE SUMMARY

```markdown
# 4.10 Complete Baseline CNN Summary

The baseline CNN consists of the following stages:

| Stage | Component | Configuration | Purpose |
|---|---|---|---|
| Input | Image | 224 × 224 × 3 | Standardized RGB input |
| Augmentation | Image transformations | Flip, rotation, zoom, contrast | Improve robustness |
| Normalization | Rescaling | 0–255 → 0–1 | Stable numerical input |
| Feature Extraction | Conv2D | 32 filters, 3×3, ReLU | Learn low-level features |
| Downsampling | MaxPooling | Pooling | Reduce spatial dimensions |
| Feature Extraction | Conv2D | 64 filters, 3×3, ReLU | Learn intermediate features |
| Downsampling | MaxPooling | Pooling | Reduce spatial dimensions |
| Feature Extraction | Conv2D | 128 filters, 3×3, ReLU | Learn higher-level features |
| Downsampling | MaxPooling | Pooling | Reduce spatial dimensions |
| Feature Aggregation | GlobalAveragePooling2D | — | Compact feature representation |
| Regularization | Dropout | 0.40 | Reduce overfitting |
| Classification | Dense | 64 neurons, ReLU | Learn classification representation |
| Output | Dense | 1 neuron, Sigmoid | Binary probability |

                    CASTING IMAGE
                          │
                          ▼
                 224 × 224 × 3
                          │
                          ▼
              ┌────────────────────┐
              │ PREPROCESSING      │
              │ Resize + Normalize │
              └─────────┬──────────┘
                        │
                        ▼
              ┌────────────────────┐
              │ DATA AUGMENTATION  │
              │ Flip / Rotate      │
              │ Zoom / Contrast    │
              └─────────┬──────────┘
                        │
                        ▼
             ┌─────────────────────┐
             │   BASELINE CNN      │
             │                     │
             │ Conv 32             │
             │ MaxPool             │
             │ Conv 64             │
             │ MaxPool             │
             │ Conv 128            │
             │ MaxPool             │
             │ GAP                 │
             │ Dropout             │
             │ Dense 64            │
             │ Sigmoid             │
             └─────────┬───────────┘
                       │
                       ▼
                BASELINE RESULT
                       │
                       │
                 IMPROVEMENT
                       │
                       ▼
             ┌─────────────────────┐
             │   MobileNetV2       │
             │ Transfer Learning   │
             └─────────┬───────────┘
                       │
                       ▼
                  Fine-Tuning
                       │
                       ▼
                FINAL MODEL
                       │
             ┌─────────┴─────────┐
             ▼                   ▼
       Classification         Grad-CAM
             │                   │
             ▼                   ▼
       Defective /          Visual
       Non-defective        Explanation
             │                   │
             └─────────┬─────────┘
                       ▼
                STREAMLIT APP

# 5. Training Design Choices

The following configuration was used for the CNN training process.

| Design Decision | Selected Value | Purpose |
|---|---|---|
| Optimizer | Adam | Adaptive weight optimization |
| Learning Rate | 0.001 | Controls update size during training |
| Loss Function | Binary Cross-Entropy | Suitable for binary classification |
| Batch Size | 32 | Balance between memory usage and training stability |
| Maximum Epochs | 25 | Provides sufficient training opportunity |
| Dropout | 0.40 | Reduces overfitting |
| Data Augmentation | Flip, rotation, zoom, contrast | Improves robustness |
| Output Activation | Sigmoid | Produces binary probability |

The training configuration follows the assessment's recommended
starting configuration.

# 6. Training Callbacks

Two important callbacks were used during training.

## 6.1 EarlyStopping

Conceptually:

```python
EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)



# 7. Model Evaluation

The model was evaluated using metrics appropriate for binary
classification.

## Metrics

### Accuracy

Accuracy measures the proportion of correctly classified
samples among all evaluated samples.

### Precision

Precision measures how many samples predicted as defective
were actually defective.

### Recall

Recall measures how many actual defective samples were
successfully detected by the model.

### F1 Score

F1 Score combines precision and recall into a single metric.

### Confusion Matrix

The confusion matrix provides four outcomes:

| | Actual Non-defective | Actual Defective |
|---|---:|---:|
| Predicted Non-defective | True Negative (TN) | False Negative (FN) |
| Predicted Defective | False Positive (FP) | True Positive (TP) |

For industrial inspection, recall is particularly important
because missing an actual defect can be more problematic than
flagging an additional non-defective sample for inspection.

# 8. Advanced Experiment — MobileNetV2 Transfer Learning

After establishing the custom CNN baseline, the project was
extended using **transfer learning**.

## Architecture

```text
Input Image
224 × 224 × 3
       ↓
MobileNetV2 Backbone
       ↓
GlobalAveragePooling2D
       ↓
Dropout
       ↓
Dense(64, ReLU)
       ↓
Dense(1, Sigmoid)
       ↓
Defective Probability

# 9. Fine-Tuning

After transfer learning, the MobileNetV2-based model was
further fine-tuned for the casting dataset.

## Fine-Tuning Process

```text
Pretrained MobileNetV2
        ↓
Train classification head
        ↓
Unfreeze selected deeper layers
        ↓
Continue training with a smaller learning rate
        ↓
Adapt visual features to casting images
        ↓
Final fine-tuned model

# 10. Decision Threshold Analysis

The sigmoid output represents the estimated probability of the
defective class.

The default classification threshold is:

**0.50**

However, the optimal threshold depends on the desired balance
between precision and recall.

Therefore, multiple thresholds were evaluated using validation
data.

### Threshold Trade-off

A lower threshold generally makes the model more sensitive to
potential defects.

This can increase recall but may also increase false positives.

A higher threshold can reduce false positives but may increase
false negatives.

The threshold was therefore selected using the validation set
rather than repeatedly tuning it on the final test set.

### Important Evaluation Principle

```text
Training Data
      ↓
Model Learning

Validation Data
      ↓
Threshold Selection

Test Data
      ↓
Final Unseen Evaluation

# 11. Explainable AI — Grad-CAM

The final model was extended with **Grad-CAM
(Gradient-weighted Class Activation Mapping)**.

Grad-CAM provides a visual explanation of the model's decision
by highlighting image regions that contributed strongly to the
prediction.

## Process

```text
Input Casting Image
        ↓
Fine-Tuned MobileNetV2
        ↓
Prediction
        +
Convolutional Feature Maps
        ↓
Gradient Calculation
        ↓
Grad-CAM Heatmap
        ↓
Overlay on Original Image

# 12. Streamlit Application

The final trained model was integrated into an interactive
Streamlit application.

## Application Architecture

```text
User
  ↓
Upload Casting Image
  ↓
Image Preprocessing
  ↓
Fine-Tuned MobileNetV2
  ↓
Defective Probability
  ↓
Decision Threshold
  ↓
Classification
  ↓
Grad-CAM
  ↓
Inspection Dashboard

# 13. Final End-to-End Architecture

The complete project architecture combines the assessment
baseline with the advanced experiments and deployment layer.

```text
                         CASTING IMAGE
                               │
                               ▼
                    ┌─────────────────────┐
                    │ IMAGE PREPROCESSING │
                    │                     │
                    │ Resize 224 × 224    │
                    │ RGB Conversion      │
                    │ Normalization       │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ DATA AUGMENTATION   │
                    │                     │
                    │ Flip                │
                    │ Rotation            │
                    │ Zoom                │
                    │ Contrast            │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │   BASELINE CNN      │
                    │                     │
                    │ Conv2D 32           │
                    │ MaxPooling          │
                    │ Conv2D 64           │
                    │ MaxPooling          │
                    │ Conv2D 128          │
                    │ MaxPooling          │
                    │ GlobalAveragePool   │
                    │ Dropout 0.40        │
                    │ Dense 64 + ReLU     │
                    │ Dense 1 + Sigmoid   │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ BASELINE EVALUATION │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ TRANSFER LEARNING   │
                    │     MobileNetV2     │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │     FINE-TUNING     │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │    FINAL MODEL      │
                    └──────────┬──────────┘
                               │
                    ┌──────────┴──────────┐
                    ▼                     ▼
             CLASSIFICATION            Grad-CAM
                    │                     │
                    ▼                     ▼
          Defective Probability     Heatmap
                    │                     │
                    └──────────┬──────────┘
                               ▼
                    ┌─────────────────────┐
                    │  STREAMLIT APP       │
                    │                     │
                    │ Prediction          │
                    │ Confidence          │
                    │ Visualization       │
                    │ Explanation         │
                    └─────────────────────┘

# 14. Design Decision Table

The following table summarizes the major design decisions made
throughout the project.

| Category | Decision | Selected Choice | Reason |
|---|---|---|---|
| Problem | Classification type | Binary classification | Two target classes |
| Input | Image size | 224 × 224 × 3 | Fixed input and computational balance |
| Input | Color format | RGB | Preserves color information |
| Preprocessing | Normalization | 0–255 → 0–1 | Stable numerical input |
| Augmentation | Transformations | Flip, rotation, zoom, contrast | Improve robustness |
| Baseline | Architecture | Custom CNN | Establish a transparent baseline |
| CNN | Filters | 32 → 64 → 128 | Increasing feature capacity |
| CNN | Kernel | 3 × 3 | Local feature extraction |
| CNN | Activation | ReLU | Introduces non-linearity |
| CNN | Pooling | MaxPooling2D | Reduce spatial dimensions |
| CNN | Feature aggregation | GlobalAveragePooling2D | Compact representation |
| Regularization | Dropout | 0.40 | Reduce overfitting |
| Classifier | Hidden layer | Dense(64, ReLU) | Transform extracted features |
| Output | Output layer | Dense(1, Sigmoid) | Binary probability |
| Loss | Loss function | Binary Cross-Entropy | Binary classification |
| Optimizer | Optimizer | Adam | Adaptive optimization |
| Training | Batch size | 32 | Memory/training balance |
| Training | Maximum epochs | 25 | Training budget |
| Training | Early stopping | Yes | Reduce overfitting/unnecessary training |
| Training | LR reduction | ReduceLROnPlateau | Improve convergence |
| Advanced model | Backbone | MobileNetV2 | Transfer learning |
| Advanced model | Fine-tuning | Yes | Adapt pretrained features |
| Decision | Threshold | Validation-selected | Balance precision and recall |
| Explainability | Method | Grad-CAM | Visualize influential regions |
| Deployment | Framework | Streamlit | Interactive inference application |


# Casting Quality Inspection
## Architecture and Design Decisions

### Binary Image Classification Using Deep Learning

**Objective:**
Develop a deep learning system to classify casting product images into two categories:

- **Class 0 → Non-defective**
- **Class 1 → Defective**

### Project Approach

This project follows a progressive deep learning approach:

1. Dataset preparation and preprocessing
2. Custom CNN baseline
3. Model training and evaluation
4. Transfer learning using MobileNetV2
5. Fine-tuning
6. Threshold analysis
7. Grad-CAM explainability
8. Streamlit deployment

The custom CNN serves as the baseline architecture required for the
assessment. MobileNetV2, fine-tuning, threshold optimization,
Grad-CAM, and Streamlit deployment are additional improvements
implemented in the project.

# 1. Dataset

The project uses the **Casting Product Image Data for Quality Inspection**
dataset for binary image classification.

The dataset contains two categories of casting images:

| Label | Class | Description |
|---|---|---|
| 0 | Non-defective | Casting product without a visible defect |
| 1 | Defective | Casting product containing a defect |

### Dataset Organization

The images are organized into folders representing the two classes:

- `ok_front` → Non-defective
- `def_front` → Defective

The objective is to learn visual patterns that allow the model to
distinguish between defective and non-defective casting products.

# 2. Input Representation

## Image Size

The input images are resized to:

**224 × 224 × 3**

where:

- `224` = image height
- `224` = image width
- `3` = RGB color channels

### Why 224 × 224?

A fixed input size is required so that all images can be processed
by the neural network using a consistent tensor shape.

The size of **224 × 224** provides a practical balance between:

- Image detail
- Computational cost
- Memory requirements
- Compatibility with the MobileNetV2 architecture used later

### Pixel Representation

The original image pixels are represented in the range:

**0–255**

During preprocessing, the values are normalized to:

**0–1**

This provides a suitable numerical range for neural network training.

# 3. Data Augmentation

Data augmentation is applied during training to expose the model
to slightly different versions of the same images.

The augmentation operations used in the project include:

- Horizontal flipping
- Small rotations
- Zoom
- Contrast variation

### Purpose of Data Augmentation

Data augmentation helps the model become less dependent on the
exact appearance or orientation of individual training images.

It can improve:

- Generalization
- Robustness to image variation
- Resistance to overfitting

The augmented images are used during training, while validation
and test images are kept separate for unbiased evaluation.

# 4. Baseline CNN Architecture

The baseline model follows the CNN architecture specified for
the assessment.

The purpose of the baseline is to understand how a conventional
CNN performs on the casting defect classification task before
introducing transfer learning.

## Complete Baseline Architecture

```text
Input Image
224 × 224 × 3
       ↓
Data Augmentation
       ↓
Rescaling
0–255 → 0–1
       ↓
Conv2D
32 filters, 3×3 kernel, ReLU
       ↓
MaxPooling2D
       ↓
Conv2D
64 filters, 3×3 kernel, ReLU
       ↓
MaxPooling2D
       ↓
Conv2D
128 filters, 3×3 kernel, ReLU
       ↓
MaxPooling2D
       ↓
GlobalAveragePooling2D
       ↓
Dropout
0.40
       ↓
Dense
64 neurons, ReLU
       ↓
Dense
1 neuron, Sigmoid
       ↓
Defective Probability

Raw Image
   ↓
Edges and simple patterns
   ↓
Intermediate visual patterns
   ↓
Complex visual patterns
   ↓
Compact feature representation
   ↓
Binary classification


## 4.1 Convolutional Layer — 32 Filters

The first convolutional layer uses:

**32 filters with a 3 × 3 kernel**

Conceptually:

```python
Conv2D(32, 3, activation="relu")


## 4.2 Convolutional Layer — 64 Filters

The second convolutional layer uses:

**64 filters with a 3 × 3 kernel**

Conceptually:

```python
Conv2D(64, 3, activation="relu")





## 4.3 Convolutional Layer — 128 Filters

The third convolutional layer uses:

**128 filters with a 3 × 3 kernel**
Conceptually:

```python
Conv2D(128, 3, activation="relu")



## 4.4 ReLU Activation

The convolutional layers and hidden Dense layer use the
**ReLU (Rectified Linear Unit)** activation function.

Conceptually:

```python
activation="relu"

## 4.7 Dropout

The baseline architecture uses:

```python
Dropout(0.40)
## 4.5 MaxPooling

MaxPooling is applied after each convolutional block.

Conceptually:

```python
MaxPooling2D()

## 4.8 Dense Layer

After the convolutional feature extraction stage, the model
uses a fully connected Dense layer:

```python
Dense(64, activation="relu")



The assessment specifies a single sigmoid output and an initial 0.50 threshold. :contentReference[oaicite:9]{index=9}

---

# CELL 15 — COMPLETE ARCHITECTURE SUMMARY

```markdown
# 4.10 Complete Baseline CNN Summary

The baseline CNN consists of the following stages:

| Stage | Component | Configuration | Purpose |
|---|---|---|---|
| Input | Image | 224 × 224 × 3 | Standardized RGB input |
| Augmentation | Image transformations | Flip, rotation, zoom, contrast | Improve robustness |
| Normalization | Rescaling | 0–255 → 0–1 | Stable numerical input |
| Feature Extraction | Conv2D | 32 filters, 3×3, ReLU | Learn low-level features |
| Downsampling | MaxPooling | Pooling | Reduce spatial dimensions |
| Feature Extraction | Conv2D | 64 filters, 3×3, ReLU | Learn intermediate features |
| Downsampling | MaxPooling | Pooling | Reduce spatial dimensions |
| Feature Extraction | Conv2D | 128 filters, 3×3, ReLU | Learn higher-level features |
| Downsampling | MaxPooling | Pooling | Reduce spatial dimensions |
| Feature Aggregation | GlobalAveragePooling2D | — | Compact feature representation |
| Regularization | Dropout | 0.40 | Reduce overfitting |
| Classification | Dense | 64 neurons, ReLU | Learn classification representation |
| Output | Dense | 1 neuron, Sigmoid | Binary probability |

                    CASTING IMAGE
                          │
                          ▼
                 224 × 224 × 3
                          │
                          ▼
              ┌────────────────────┐
              │ PREPROCESSING      │
              │ Resize + Normalize │
              └─────────┬──────────┘
                        │
                        ▼
              ┌────────────────────┐
              │ DATA AUGMENTATION  │
              │ Flip / Rotate      │
              │ Zoom / Contrast    │
              └─────────┬──────────┘
                        │
                        ▼
             ┌─────────────────────┐
             │   BASELINE CNN      │
             │                     │
             │ Conv 32             │
             │ MaxPool             │
             │ Conv 64             │
             │ MaxPool             │
             │ Conv 128            │
             │ MaxPool             │
             │ GAP                 │
             │ Dropout             │
             │ Dense 64            │
             │ Sigmoid             │
             └─────────┬───────────┘
                       │
                       ▼
                BASELINE RESULT
                       │
                       │
                 IMPROVEMENT
                       │
                       ▼
             ┌─────────────────────┐
             │   MobileNetV2       │
             │ Transfer Learning   │
             └─────────┬───────────┘
                       │
                       ▼
                  Fine-Tuning
                       │
                       ▼
                FINAL MODEL
                       │
             ┌─────────┴─────────┐
             ▼                   ▼
       Classification         Grad-CAM
             │                   │
             ▼                   ▼
       Defective /          Visual
       Non-defective        Explanation
             │                   │
             └─────────┬─────────┘
                       ▼
                STREAMLIT APP

# 5. Training Design Choices

The following configuration was used for the CNN training process.

| Design Decision | Selected Value | Purpose |
|---|---|---|
| Optimizer | Adam | Adaptive weight optimization |
| Learning Rate | 0.001 | Controls update size during training |
| Loss Function | Binary Cross-Entropy | Suitable for binary classification |
| Batch Size | 32 | Balance between memory usage and training stability |
| Maximum Epochs | 25 | Provides sufficient training opportunity |
| Dropout | 0.40 | Reduces overfitting |
| Data Augmentation | Flip, rotation, zoom, contrast | Improves robustness |
| Output Activation | Sigmoid | Produces binary probability |

The training configuration follows the assessment's recommended
starting configuration.

# 6. Training Callbacks

Two important callbacks were used during training.

## 6.1 EarlyStopping

Conceptually:

```python
EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)



# 7. Model Evaluation

The model was evaluated using metrics appropriate for binary
classification.

## Metrics

### Accuracy

Accuracy measures the proportion of correctly classified
samples among all evaluated samples.

### Precision

Precision measures how many samples predicted as defective
were actually defective.

### Recall

Recall measures how many actual defective samples were
successfully detected by the model.

### F1 Score

F1 Score combines precision and recall into a single metric.

### Confusion Matrix

The confusion matrix provides four outcomes:

| | Actual Non-defective | Actual Defective |
|---|---:|---:|
| Predicted Non-defective | True Negative (TN) | False Negative (FN) |
| Predicted Defective | False Positive (FP) | True Positive (TP) |

For industrial inspection, recall is particularly important
because missing an actual defect can be more problematic than
flagging an additional non-defective sample for inspection.

# 8. Advanced Experiment — MobileNetV2 Transfer Learning

After establishing the custom CNN baseline, the project was
extended using **transfer learning**.

## Architecture

```text
Input Image
224 × 224 × 3
       ↓
MobileNetV2 Backbone
       ↓
GlobalAveragePooling2D
       ↓
Dropout
       ↓
Dense(64, ReLU)
       ↓
Dense(1, Sigmoid)
       ↓
Defective Probability

# 9. Fine-Tuning

After transfer learning, the MobileNetV2-based model was
further fine-tuned for the casting dataset.

## Fine-Tuning Process

```text
Pretrained MobileNetV2
        ↓
Train classification head
        ↓
Unfreeze selected deeper layers
        ↓
Continue training with a smaller learning rate
        ↓
Adapt visual features to casting images
        ↓
Final fine-tuned model

# 10. Decision Threshold Analysis

The sigmoid output represents the estimated probability of the
defective class.

The default classification threshold is:

**0.50**

However, the optimal threshold depends on the desired balance
between precision and recall.

Therefore, multiple thresholds were evaluated using validation
data.

### Threshold Trade-off

A lower threshold generally makes the model more sensitive to
potential defects.

This can increase recall but may also increase false positives.

A higher threshold can reduce false positives but may increase
false negatives.

The threshold was therefore selected using the validation set
rather than repeatedly tuning it on the final test set.

### Important Evaluation Principle

```text
Training Data
      ↓
Model Learning

Validation Data
      ↓
Threshold Selection

Test Data
      ↓
Final Unseen Evaluation

# 11. Explainable AI — Grad-CAM

The final model was extended with **Grad-CAM
(Gradient-weighted Class Activation Mapping)**.

Grad-CAM provides a visual explanation of the model's decision
by highlighting image regions that contributed strongly to the
prediction.

## Process

```text
Input Casting Image
        ↓
Fine-Tuned MobileNetV2
        ↓
Prediction
        +
Convolutional Feature Maps
        ↓
Gradient Calculation
        ↓
Grad-CAM Heatmap
        ↓
Overlay on Original Image

# 12. Streamlit Application

The final trained model was integrated into an interactive
Streamlit application.

## Application Architecture

```text
User
  ↓
Upload Casting Image
  ↓
Image Preprocessing
  ↓
Fine-Tuned MobileNetV2
  ↓
Defective Probability
  ↓
Decision Threshold
  ↓
Classification
  ↓
Grad-CAM
  ↓
Inspection Dashboard

# 13. Final End-to-End Architecture

The complete project architecture combines the assessment
baseline with the advanced experiments and deployment layer.

```text
                         CASTING IMAGE
                               │
                               ▼
                    ┌─────────────────────┐
                    │ IMAGE PREPROCESSING │
                    │                     │
                    │ Resize 224 × 224    │
                    │ RGB Conversion      │
                    │ Normalization       │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ DATA AUGMENTATION   │
                    │                     │
                    │ Flip                │
                    │ Rotation            │
                    │ Zoom                │
                    │ Contrast            │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │   BASELINE CNN      │
                    │                     │
                    │ Conv2D 32           │
                    │ MaxPooling          │
                    │ Conv2D 64           │
                    │ MaxPooling          │
                    │ Conv2D 128          │
                    │ MaxPooling          │
                    │ GlobalAveragePool   │
                    │ Dropout 0.40        │
                    │ Dense 64 + ReLU     │
                    │ Dense 1 + Sigmoid   │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ BASELINE EVALUATION │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ TRANSFER LEARNING   │
                    │     MobileNetV2     │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │     FINE-TUNING     │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │    FINAL MODEL      │
                    └──────────┬──────────┘
                               │
                    ┌──────────┴──────────┐
                    ▼                     ▼
             CLASSIFICATION            Grad-CAM
                    │                     │
                    ▼                     ▼
          Defective Probability     Heatmap
                    │                     │
                    └──────────┬──────────┘
                               ▼
                    ┌─────────────────────┐
                    │  STREAMLIT APP       │
                    │                     │
                    │ Prediction          │
                    │ Confidence          │
                    │ Visualization       │
                    │ Explanation         │
                    └─────────────────────┘

# 14. Design Decision Table

The following table summarizes the major design decisions made
throughout the project.

| Category | Decision | Selected Choice | Reason |
|---|---|---|---|
| Problem | Classification type | Binary classification | Two target classes |
| Input | Image size | 224 × 224 × 3 | Fixed input and computational balance |
| Input | Color format | RGB | Preserves color information |
| Preprocessing | Normalization | 0–255 → 0–1 | Stable numerical input |
| Augmentation | Transformations | Flip, rotation, zoom, contrast | Improve robustness |
| Baseline | Architecture | Custom CNN | Establish a transparent baseline |
| CNN | Filters | 32 → 64 → 128 | Increasing feature capacity |
| CNN | Kernel | 3 × 3 | Local feature extraction |
| CNN | Activation | ReLU | Introduces non-linearity |
| CNN | Pooling | MaxPooling2D | Reduce spatial dimensions |
| CNN | Feature aggregation | GlobalAveragePooling2D | Compact representation |
| Regularization | Dropout | 0.40 | Reduce overfitting |
| Classifier | Hidden layer | Dense(64, ReLU) | Transform extracted features |
| Output | Output layer | Dense(1, Sigmoid) | Binary probability |
| Loss | Loss function | Binary Cross-Entropy | Binary classification |
| Optimizer | Optimizer | Adam | Adaptive optimization |
| Training | Batch size | 32 | Memory/training balance |
| Training | Maximum epochs | 25 | Training budget |
| Training | Early stopping | Yes | Reduce overfitting/unnecessary training |
| Training | LR reduction | ReduceLROnPlateau | Improve convergence |
| Advanced model | Backbone | MobileNetV2 | Transfer learning |
| Advanced model | Fine-tuning | Yes | Adapt pretrained features |
| Decision | Threshold | Validation-selected | Balance precision and recall |
| Explainability | Method | Grad-CAM | Visualize influential regions |
| Deployment | Framework | Streamlit | Interactive inference application |
# 15. Experimental Progression

The project was developed progressively rather than using a
single model from the beginning.

```text
Experiment 1
Custom CNN Baseline
        ↓
Establish baseline performance
        ↓
Experiment 2
MobileNetV2 Transfer Learning
        ↓
Investigate pretrained visual features
        ↓
Experiment 3
Fine-Tuning
        ↓
Adapt pretrained features to casting images
        ↓
Experiment 4
Threshold Analysis
        ↓
Study precision-recall trade-off
        ↓
Experiment 5
Grad-CAM
        ↓
Add model explainability
        ↓
Experiment 6
Streamlit
        ↓
Deploy the complete inference pipeline


# 16. Final Project Summary

The project developed an end-to-end casting quality inspection
system for binary classification of casting images.

The development began with a custom CNN baseline containing
three convolutional stages with increasing filter counts of
32, 64, and 128.

The baseline architecture was then extended using MobileNetV2
transfer learning and fine-tuning to investigate whether
pretrained visual representations could improve the model.

Threshold analysis was performed using validation data to study
the trade-off between precision and recall.

Grad-CAM was then integrated to provide visual explanations of
model predictions.

Finally, the trained model was deployed through a Streamlit
application that allows users to upload casting images and
receive:

- Classification
- Defective probability
- Confidence
- Grad-CAM visualization

The final system therefore combines:

**CNN Architecture**

→ **Transfer Learning**

→ **Fine-Tuning**

→ **Threshold Optimization**

→ **Explainable AI**

→ **Streamlit Deployment**

This provides both an academic experimental workflow and a
practical computer-vision inspection application.

# 17. Evaluation Strategy

The dataset was separated into training, validation, and test
sets to prevent the final evaluation from being influenced by
model development decisions.

The evaluation workflow was:

```text
Training Set
     ↓
Model Training
     ↓
Validation Set
     ↓
Model Selection / Threshold Analysis
     ↓
Threshold Locked
     ↓
Test Set
     ↓
Final Unseen Evaluation




# 18. Validation Results

The fine-tuned MobileNetV2 model achieved the following
performance on the validation set containing **195 samples**.

| Metric | Result |
|---|---:|
| Accuracy | 97.44% |
| Precision | 100.00% |
| Recall | 95.73% |
| F1 Score | 97.82% |

### Validation Confusion Matrix

| | Actual Non-defective | Actual Defective |
|---|---:|---:|
| Predicted Non-defective | 78 | 5 |
| Predicted Defective | 0 | 112 |

Therefore:

- **True Negatives (TN): 78**
- **False Positives (FP): 0**
- **False Negatives (FN): 5**
- **True Positives (TP): 112**

### Interpretation

The model correctly identified all 112 defective samples
that it predicted as defective, resulting in a precision of
100% on this validation set.

The model missed 5 defective samples, resulting in a recall
of 95.73%.

There were no false positives in this validation result.

# 19. Confusion Matrix Analysis

The confusion matrix provides a more detailed view of the
classification behavior.

```text
                         ACTUAL
                  Non-defective   Defective
                ┌────────────────────────────┐
Predicted       │                            │
Non-defective   │      TN = 78   FN = 5     │
                │                            │
Defective       │      FP = 0    TP = 112   │
                │                            │
                └────────────────────────────┘

# 20. Classification Threshold Analysis

The model produces a probability for the defective class.

Instead of assuming that the default threshold of 0.50 is
always optimal, multiple thresholds were evaluated using
validation data.

The objective was to understand the trade-off between:

- Precision
- Recall
- F1 Score
- False Positives
- False Negatives

### Example Threshold Results

| Threshold | Accuracy | Precision | Recall | F1 | FP | FN |
|---:|---:|---:|---:|---:|---:|---:|
| 0.30 | 0.6008 | 0.6008 | 1.0000 | 0.7506 | 519 | 0 |
| 0.40 | 0.6008 | 0.6008 | 1.0000 | 0.7506 | 519 | 0 |
| 0.50 | 0.7308 | 0.7175 | 0.9104 | 0.8025 | 280 | 70 |
| 0.60 | 0.7223 | 0.7877 | 0.7362 | 0.7611 | 155 | 206 |

### Observation

Lowering the threshold makes the model more sensitive to
potential defects.

For example, at a threshold of 0.30, recall reached 100%,
but the number of false positives increased substantially.

Increasing the threshold reduces false positives but can
increase false negatives.

Therefore, threshold selection is a trade-off between
detecting defects and avoiding unnecessary rejection of
non-defective products.

# 21. Why Decision Threshold Matters

The sigmoid output of the model is a probability-like score
for the defective class.

For example:

```text
Model output = 0.82

# 22. Model Development Progression

The project was developed through multiple stages.

## Stage 1 — Custom CNN Baseline

The first model followed the assessment architecture:

```text
Conv32
  ↓
MaxPooling
  ↓
Conv64
  ↓
MaxPooling
  ↓
Conv128
  ↓
MaxPooling
  ↓
GlobalAveragePooling
  ↓
Dropout
  ↓
Dense64
  ↓
Sigmoid

# 23. Final Test Evaluation

After model development and validation-based threshold analysis,
the selected model and decision threshold were evaluated on
the previously unseen test set.

The final test results will be reported using:

| Metric | Final Test Result |
|---|---:|
| Accuracy | To be filled |
| Precision | To be filled |
| Recall | To be filled |
| F1 Score | To be filled |
| True Negatives | To be filled |
| False Positives | To be filled |
| False Negatives | To be filled |
| True Positives | To be filled |

### Important

The test set is not used to select the model or tune the
decision threshold.

It is reserved for the final evaluation so that the reported
performance provides a more reliable estimate of performance
on unseen images.

# 24. Application Validation

The trained model was integrated into the Streamlit
application and tested using representative images from both
classes.

### Test Categories

1. `def_front` — known defective casting images
2. `ok_front` — known non-defective casting images

### Application Workflow

```text
Upload Image
     ↓
Convert to RGB
     ↓
Resize to 224 × 224
     ↓
Model Prediction
     ↓
Defective Probability
     ↓
Classification
     ↓
Grad-CAM
     ↓
Display Result

# 25. Limitations

Although the model demonstrates strong validation performance,
several limitations should be considered.

### 1. Dataset Dependence

The model is trained on a specific casting image dataset.
Performance may change when images come from different
manufacturing environments, cameras, lighting conditions, or
casting types.

### 2. False Negatives

Even with high recall, some defective samples may be missed.
In an industrial environment, these cases require particular
attention.

### 3. Grad-CAM Interpretation

Grad-CAM provides an explanation of model attention, but it
does not guarantee that the highlighted region corresponds
exactly to the physical defect.

### 4. Computational Environment

The current development environment uses CPU-based TensorFlow
on native Windows. Training and inference performance may
differ on GPU-enabled environments.

### 5. Real-World Validation

The Streamlit application has been tested using dataset images.
Additional testing with real production-line images would be
required before deployment in an actual manufacturing
environment.

# 26. Future Work

Several improvements could be explored in future versions.

### 1. Larger and More Diverse Dataset

Include casting images from multiple production environments,
lighting conditions, cameras, and product types.

### 2. Defect Localization

Move beyond image-level classification toward object detection
or segmentation to identify the exact defect location.

### 3. Real-Time Inspection

Integrate the model with an industrial camera for real-time
inspection.

### 4. Edge Deployment

Investigate deployment on edge devices for low-latency
manufacturing inspection.

### 5. Model Monitoring

Track model performance after deployment and detect changes
in image distributions or prediction behavior.

### 6. Human-in-the-Loop Inspection

Allow uncertain predictions to be reviewed by a human inspector
and use the verified samples for future model improvement.

### 7. Additional Explainability

Compare Grad-CAM with other explainability techniques to
evaluate the consistency of model explanations.

# 27. Conclusion

This project developed an end-to-end deep learning system for
casting quality inspection.

The development began with a transparent custom CNN baseline
following the required assessment architecture.

The system was then extended through:

**Custom CNN**
→ **MobileNetV2 Transfer Learning**
→ **Fine-Tuning**
→ **Threshold Analysis**
→ **Grad-CAM Explainability**
→ **Streamlit Deployment**

The final application accepts a casting image and produces
a binary quality prediction together with a defective
probability and Grad-CAM visualization.

The project therefore demonstrates the complete workflow of
a computer vision system, from dataset preparation and CNN
architecture design to model improvement, evaluation,
explainability, and application deployment.

# 28. Experimental Comparison

The project was developed through a controlled progression
from a custom CNN baseline to a transfer-learning-based model.

The purpose of the comparison is to investigate whether using
a pretrained convolutional backbone provides an advantage over
learning visual features from scratch.

## Experiment A — Custom CNN

The baseline CNN uses:

```text
Conv2D 32
    ↓
MaxPooling
    ↓
Conv2D 64
    ↓
MaxPooling
    ↓
Conv2D 128
    ↓
MaxPooling
    ↓
GlobalAveragePooling
    ↓
Dropout
    ↓
Dense 64
    ↓
Sigmoid

# 29. Model Comparison

The following table is used to compare the baseline CNN and
the improved MobileNetV2-based model.

| Metric | Custom CNN Baseline | MobileNetV2 / Fine-Tuned |
|---|---:|---:|
| Accuracy | To be filled | To be filled |
| Precision | To be filled | To be filled |
| Recall | To be filled | To be filled |
| F1 Score | To be filled | To be filled |
| Training Approach | From scratch | Transfer learning + fine-tuning |
| Explainability | Not included in baseline | Grad-CAM |
| Deployment | — | Streamlit |

The comparison will be completed using the corresponding
evaluation results from the experiments.

The objective is not simply to maximize one metric, but to
understand how the architectural change affects the overall
precision-recall trade-off and generalization performance.

# 30. Analysis of the Architectural Improvement

The custom CNN provides an important baseline because its
architecture is transparent and its feature representations
are learned entirely from the casting dataset.

However, the amount of data available for training may limit
the ability of a CNN trained from scratch to learn highly
generalizable visual features.

MobileNetV2 addresses this by starting from pretrained visual
representations.

The subsequent fine-tuning stage allows the model to adapt
those representations to casting-specific visual patterns.

Therefore, the progression:

```text
Custom CNN
     ↓
Transfer Learning
     ↓
Fine-Tuning

# 31. Assessment Requirement Mapping

The implemented project addresses the major requirements of
the assessment as follows.

| Assessment Requirement | Implementation |
|---|---|
| Binary classification | Non-defective vs Defective |
| Dataset loading | Casting Product Image Data |
| Image preprocessing | Resize, RGB conversion, normalization |
| Data augmentation | Flip, rotation, zoom, contrast |
| CNN architecture | Conv2D 32 → 64 → 128 |
| Activation | ReLU |
| Pooling | MaxPooling2D |
| Feature aggregation | GlobalAveragePooling2D |
| Regularization | Dropout 0.40 |
| Output | Dense(1, Sigmoid) |
| Optimizer | Adam |
| Loss | Binary Cross-Entropy |
| Batch size | 32 |
| Training | Maximum 25 epochs with callbacks |
| Evaluation | Accuracy, Precision, Recall, F1 |
| Confusion matrix | TN, FP, FN, TP |
| Threshold analysis | Validation-based threshold analysis |
| Bonus experiment | MobileNetV2 transfer learning |
| Fine-tuning | MobileNetV2 fine-tuning |
| Explainability | Grad-CAM |
| Application | Streamlit |

# 32. Scope and Reporting Notes

The custom CNN is treated as the assessment baseline.

MobileNetV2 transfer learning, fine-tuning, threshold analysis,
Grad-CAM, and Streamlit deployment are extensions developed
as part of the project.

The reported model performance must always specify the dataset
split used for evaluation.

In particular:

- Validation results must be reported as validation results.
- Final test results must be reported separately.
- Threshold selection must be performed using validation data.
- The test set should not be repeatedly used to tune the model.

This distinction ensures that the experimental results are
reported accurately and transparently.

# 33. Final System Architecture

The final system combines the assessment baseline with the
improvements developed during the project.

```text
                         CASTING IMAGE
                               │
                               ▼
                    ┌─────────────────────┐
                    │ IMAGE PREPROCESSING │
                    │                     │
                    │ Resize 224 × 224    │
                    │ RGB Conversion      │
                    │ Normalization       │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │ DATA AUGMENTATION   │
                    │                     │
                    │ Flip                │
                    │ Rotation            │
                    │ Zoom                │
                    │ Contrast            │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │    MODEL STAGE      │
                    │                     │
                    │ Custom CNN Baseline │
                    │          ↓          │
                    │   MobileNetV2       │
                    │          ↓          │
                    │    Fine-Tuning      │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │    FINAL MODEL      │
                    └──────────┬──────────┘
                               │
                    ┌──────────┴──────────┐
                    │                     │
                    ▼                     ▼
             CLASSIFICATION            Grad-CAM
                    │                     │
                    ▼                     ▼
          Defective Probability      Heatmap
                    │                     │
                    └──────────┬──────────┘
                               ▼
                    ┌─────────────────────┐
                    │  STREAMLIT APP       │
                    │                     │
                    │ Prediction          │
                    │ Confidence          │
                    │ Visualization       │
                    │ Explanation         │
                    └─────────────────────┘

# 34. Final Model Selection

The project evaluates a progression of models rather than
selecting a model solely based on a single accuracy value.

The selection process considers:

- Accuracy
- Precision
- Recall
- F1 Score
- Confusion matrix
- False positives
- False negatives
- Generalization on unseen data
- Explainability
- Practical deployment requirements

The custom CNN serves as the assessment baseline.

The MobileNetV2-based model is investigated as an advanced
architecture using transfer learning and fine-tuning.

The final model is selected based on its overall performance
and suitability for the casting quality inspection task.

# 35. Results

The project evaluates the developed models using standard
binary classification metrics.

The following results are reported from the corresponding
training and evaluation notebooks.

## Baseline CNN

| Metric | Result |
|---|---:|
| Accuracy | Refer to baseline evaluation |
| Precision | Refer to baseline evaluation |
| Recall | Refer to baseline evaluation |
| F1 Score | Refer to baseline evaluation |

## MobileNetV2 / Fine-Tuned Model

| Metric | Result |
|---|---:|
| Accuracy | Refer to final evaluation |
| Precision | Refer to final evaluation |
| Recall | Refer to final evaluation |
| F1 Score | Refer to final evaluation |

### Validation Result Already Obtained

During validation, the fine-tuned model achieved:

- Accuracy: **97.44%**
- Precision: **100.00%**
- Recall: **95.73%**
- F1 Score: **97.82%**

Validation confusion matrix:

- True Negatives: **78**
- False Positives: **0**
- False Negatives: **5**
- True Positives: **112**

These values are explicitly identified as validation results
and should not be presented as final test-set performance.

# 36. Threshold Analysis

The model produces a continuous probability for the defective
class.

The default threshold of 0.50 was investigated along with
alternative thresholds.

The purpose of threshold analysis was to understand the
relationship between:

- Precision
- Recall
- F1 Score
- False Positives
- False Negatives

### Observed Trade-off

A lower threshold increases the sensitivity of the classifier
to potential defects.

However, lowering the threshold can also increase the number
of non-defective images incorrectly classified as defective.

Conversely, increasing the threshold can reduce false positives
but may increase false negatives.

Therefore, the decision threshold should be selected using
validation data according to the practical requirements of
the inspection system.

# 37. Explainability Analysis

Grad-CAM was incorporated into the final system to provide
visual explanations of model predictions.

## Explanation Pipeline

```text
Casting Image
     ↓
Fine-Tuned Model
     ↓
Prediction
     ↓
Convolutional Feature Maps
     ↓
Gradients
     ↓
Grad-CAM Heatmap
     ↓
Overlay with Original Image

# 38. Streamlit Application

The final model was integrated into a Streamlit application
to provide an interactive inspection interface.

## Application Workflow

```text
User
 ↓
Upload Casting Image
 ↓
Image Preprocessing
 ↓
Fine-Tuned MobileNetV2
 ↓
Defective Probability
 ↓
Classification
 ↓
Grad-CAM
 ↓
Inspection Result

# 39. Project Strengths

The project provides several important strengths:

### 1. Clear CNN Baseline

The custom CNN provides a transparent architecture in which
each major component can be explained.

### 2. Transfer Learning

MobileNetV2 provides a stronger pretrained visual feature
extractor for comparison with the baseline.

### 3. Fine-Tuning

Fine-tuning allows the pretrained representation to adapt
to the casting inspection domain.

### 4. Threshold Analysis

The project considers the precision-recall trade-off rather
than relying only on accuracy.

### 5. Explainable AI

Grad-CAM provides visual insight into model predictions.

### 6. Interactive Application

The Streamlit application converts the trained model into
an accessible inspection tool.

### 7. End-to-End Workflow

The project covers the complete pipeline from dataset
preparation to deployment.

# 40. Limitations

Despite the strong development results, the system has several
limitations.

## Dataset Limitation

The model is trained and evaluated using the available
casting image dataset. Its performance may change when
presented with images from different manufacturing conditions.

## Domain Shift

Changes in:

- Camera
- Lighting
- Background
- Casting type
- Image quality
- Production environment

may affect model performance.

## False Negatives

A defective casting that is classified as non-defective
represents a false negative and may be important in an
industrial inspection scenario.

## Grad-CAM Limitation

Grad-CAM provides model-attribution information but does not
guarantee an exact physical defect location.

## Real-World Validation

The application has been tested using dataset images.
Additional validation with real production-line images would
be required before industrial deployment.

## Hardware Limitation

The current development environment uses CPU-based TensorFlow
on native Windows. Training time may be reduced significantly
on a suitable GPU-enabled environment.

# 41. Future Work

The project can be extended in several directions.

## 1. Defect Localization

Instead of only predicting whether a casting is defective,
future versions could identify the exact location of the defect.

Possible approaches include:

- Object detection
- Semantic segmentation
- Instance segmentation

## 2. Larger Dataset

Training on a larger and more diverse dataset could improve
generalization.

## 3. Real-Time Inspection

The model could be connected to an industrial camera for
continuous inspection.

## 4. Edge Deployment

The model could be optimized for deployment on edge devices
located near the production line.

## 5. Human-in-the-Loop System

Low-confidence predictions could be sent to a human inspector
for verification.

Verified results could later be incorporated into model
improvement workflows.

## 6. Model Monitoring

A production system could monitor prediction distributions,
data drift, and model performance over time.

## 7. Advanced Explainability

Grad-CAM could be compared with other explainability
techniques to study the consistency of model explanations.

# 42. Conclusion

This project developed an end-to-end deep learning system for
casting quality inspection.

The project began with a transparent custom CNN baseline
designed for binary classification of defective and
non-defective casting images.

The architecture was then extended through MobileNetV2
transfer learning and fine-tuning.

Threshold analysis was performed to study the trade-off
between precision and recall.

Grad-CAM was incorporated to provide visual explanations
of model predictions.

Finally, the trained model was integrated into a Streamlit
application that allows users to upload casting images and
receive classification results, confidence information,
and visual explanations.

The overall development pipeline can therefore be summarized
as:

```text
Dataset
   ↓
Preprocessing
   ↓
Custom CNN Baseline
   ↓
Evaluation
   ↓
MobileNetV2 Transfer Learning
   ↓
Fine-Tuning
   ↓
Threshold Analysis
   ↓
Grad-CAM Explainability
   ↓
Streamlit Deployment

# 43. Final Takeaway

The key contribution of the project is not only the final
classification accuracy.

The project demonstrates a complete methodology:

**Understand → Build → Compare → Improve → Explain → Deploy**

- **Understand:** Analyze the casting dataset and define the
  binary classification problem.
- **Build:** Develop the custom CNN baseline.
- **Compare:** Evaluate the baseline against a MobileNetV2-based
  transfer-learning approach.
- **Improve:** Fine-tune the pretrained model and analyze the
  classification threshold.
- **Explain:** Use Grad-CAM to visualize influential image
  regions.
- **Deploy:** Integrate the final inference pipeline into a
  Streamlit application.

